# Final project: approach tests

These tests compare the options behind the choices in `Final Project.ipynb`. Run that notebook first, because this one reads its stored results. There is no ground truth for the robot's position, so the detector of the main notebook serves as the reference for the other detectors. Its boxes were checked by eye on a sample of frames.

#### Setup
The cell loads the stored calibration and detections, and repeats the small helper functions of the main notebook.

In [1]:
import glob
import os
import time

import cv2
import imutils
import numpy as np
from matplotlib import pyplot as plt
from sklearn import linear_model, svm
from sklearn.decomposition import PCA

RESULTS = 'results'
SEQUENCES = ['1_no_obstacle', '2_obstacle', '3_curved_bend', '4_circle']
IMAGE_SIZE = (1280, 720)
BOARD = (7, 5)
SQUARE_SIZE = 0.080
MIN_AREA = 1000

stored = np.load(os.path.join(RESULTS, 'calibration.npz'))
K1, d1, K2, d2, R, T = stored['K1'], stored['d1'], stored['K2'], stored['d2'], stored['R'], stored['T']
R1, R2, P1, P2 = stored['R1'], stored['R2'], stored['P1'], stored['P2']
map_l = cv2.initUndistortRectifyMap(K1, d1, R1, P1, IMAGE_SIZE, cv2.CV_32FC1)
map_r = cv2.initUndistortRectifyMap(K2, d2, R2, P2, IMAGE_SIZE, cv2.CV_32FC1)
FOCAL, BASELINE = P1[0, 0], -P2[0, 3] / P2[0, 0]
valid = (map_l[0] >= 0) & (map_l[0] < IMAGE_SIZE[0]) & (map_l[1] >= 0) & (map_l[1] < IMAGE_SIZE[1])
LEFT_LIMIT = max(np.flatnonzero(valid[row])[0] for row in range(300, 650))
RIGHT_LIMIT = min(np.flatnonzero(valid[row])[-1] for row in range(300, 650))

detections = {s: np.load(os.path.join(RESULTS, s + '_detections.npy')) for s in SEQUENCES}
backgrounds = {s: cv2.imread(os.path.join(RESULTS, s + '_background_left.png')) for s in SEQUENCES}
backgrounds_r = {s: cv2.imread(os.path.join(RESULTS, s + '_background_right.png')) for s in SEQUENCES}
block_matcher = cv2.StereoBM_create(numDisparities=64, blockSize=15)


def image_files(sequence, side):
    return sorted(glob.glob(os.path.join(sequence, sequence, side, '*.png')))


def rectified(path, side, maps=None):
    if maps is None:
        maps = map_l if side == 'left' else map_r
    return cv2.remap(cv2.imread(path), maps[0], maps[1], cv2.INTER_LINEAR)


def foreground(image, background, threshold=18):
    difference = np.abs(cv2.GaussianBlur(image, (5, 5), 0).astype(int) - cv2.GaussianBlur(background, (5, 5), 0).astype(int)).max(axis=2)
    return cv2.dilate(cv2.erode((difference > threshold).astype(np.uint8) * 255, None, iterations=1), None, iterations=2)


def blob_list(mask):
    contours = imutils.grab_contours(cv2.findContours(mask.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE))
    blobs = []
    for c in contours:
        x0, x1, y0, y1 = c[:, :, 0].min(), c[:, :, 0].max() + 1, c[:, :, 1].min(), c[:, :, 1].max() + 1
        blobs.append((np.sum(mask[y0:y1, x0:x1] > 0), x0, y0, x1 - x0, y1 - y0))
    return sorted(blobs, reverse=True)


def iou(a, b):
    w = min(a[0] + a[2], b[0] + b[2]) - max(a[0], b[0])
    h = min(a[1] + a[3], b[1] + b[3]) - max(a[1], b[1])
    overlap = max(w, 0) * max(h, 0)
    return overlap / (a[2] * a[3] + b[2] * b[3] - overlap)


def complete(row):
    if row[0] < MIN_AREA or np.isnan(row[5]) or row[5] <= 0:
        return False
    return LEFT_LIMIT + 2 < row[1] and row[1] + row[3] < RIGHT_LIMIT - 2 and 0.6 <= row[3] * BASELINE / row[5] <= 2.2 and 0.8 <= row[4] * BASELINE / row[5] <= 2.2

#### 1. Calibration variants
Three corner detectors are compared: `cv2.findChessboardCorners` from Class 4, the same corners refined by `cv2.cornerSubPix`, and `cv2.findChessboardCornersSB`. The last two aren't in the course material. The first table counts the boards each detector finds, and measures how far its corners lie from the board's planar shape: one homography is fitted to all 35 corners, and the distance from it is the error. Each detector is then used in one of two calibrations. **Two-step** calibrates each camera alone with `cv2.calibrateCamera`, and keeps those numbers fixed in `cv2.stereoCalibrate`. **Joint** lets `cv2.stereoCalibrate` refine both cameras together. Each calibration is scored by the reprojection error on the board and by the row difference of SIFT matches on real frames after rectification. The negative share is the share of scene matches with a negative disparity, which is physically impossible for a correct rectification.

In [2]:
SB_FLAGS = cv2.CALIB_CB_EXHAUSTIVE | cv2.CALIB_CB_NORMALIZE_IMAGE | cv2.CALIB_CB_ACCURACY
CRITERIA = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.01)


def corner_variants(path):
    gray = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    found = {}
    for scale in (1, 1.5, 2, 2.5, 3):
        ok, corners = cv2.findChessboardCorners(cv2.resize(gray, None, fx=scale, fy=scale), BOARD)
        if ok:
            plain = (corners.reshape(-1, 1, 2) / scale).astype(np.float32)
            refined = cv2.cornerSubPix(gray, plain.copy(), (5, 5), (-1, -1), CRITERIA)
            found.update(scale=scale, classic=plain.reshape(-1, 2), subpix=refined.reshape(-1, 2))
            break
    ok, corners = cv2.findChessboardCornersSB(gray, BOARD, flags=SB_FLAGS)
    if ok:
        found['sb'] = corners.reshape(-1, 2)
    return found


variants_file = os.path.join(RESULTS, 'corner_variants.npy')
if os.path.exists(variants_file):
    found_l, found_r = np.load(variants_file, allow_pickle=True)
else:
    found_l = [corner_variants(p) for p in image_files('0_calib', 'left')]
    found_r = [corner_variants(p) for p in image_files('0_calib', 'right')]
    store = np.empty(2, dtype=object)
    store[0], store[1] = found_l, found_r
    np.save(variants_file, store)

GRID = np.zeros((BOARD[0] * BOARD[1], 3), np.float32)
GRID[:, :2] = np.mgrid[0:BOARD[0], 0:BOARD[1]].T.reshape(-1, 2)


def ordered(corners):
    return corners[::-1] if corners[-1, 0] < corners[0, 0] else corners


def shape_error(corners):
    H, _ = cv2.findHomography(GRID[:, :2], corners, 0)
    fitted = cv2.perspectiveTransform(GRID[:, :2].reshape(-1, 1, 2), H).reshape(-1, 2)
    return np.sqrt(np.mean(np.sum((fitted - corners) ** 2, axis=1)))


print("detector | board found: left, right, pairs | distance from the board's planar shape [px]: median, worst")
for detector in ('classic', 'subpix', 'sb'):
    left, right = [f.get(detector) for f in found_l], [f.get(detector) for f in found_r]
    errors = [shape_error(c) for c in left + right if c is not None]
    pairs = sum(a is not None and b is not None for a, b in zip(left, right))
    print(f"  {detector:8s} | {sum(c is not None for c in left)}, {sum(c is not None for c in right)}, {pairs} | {np.median(errors):.2f}, {np.max(errors):.2f}")


def scene_matches(left, right):
    sift = cv2.SIFT_create(4000)
    kp1, des1 = sift.detectAndCompute(cv2.cvtColor(left, cv2.COLOR_BGR2GRAY), None)
    kp2, des2 = sift.detectAndCompute(cv2.cvtColor(right, cv2.COLOR_BGR2GRAY), None)
    good = [m for m, n in cv2.BFMatcher(cv2.NORM_L2).knnMatch(des1, des2, k=2) if m.distance < 0.7 * n.distance]
    return np.float32([kp1[m.queryIdx].pt for m in good]), np.float32([kp2[m.trainIdx].pt for m in good])


def usable(found, detector, one_scale):
    corners = found.get(detector)
    if corners is None or (one_scale and found.get('scale') != 1):
        return None
    return ordered(corners)


def planar(corners, ransac):
    if not ransac:
        return np.ones(len(corners), bool)
    return cv2.findHomography(GRID[:, :2], corners, cv2.RANSAC, 1.0)[1].ravel() == 1


def calibrate(detector, one_scale=False, ransac=True, joint=True, flags=0, pick=None):
    """Returns (K1, d1, K2, d2, R, T) with T in metres, and (stereo views, rms left, right, stereo)."""
    objects_l, images_l, objects_r, images_r, objects_s, stereo_l, stereo_r = [], [], [], [], [], [], []
    for i in range(len(found_l)) if pick is None else pick:
        a, b = usable(found_l[i], detector, one_scale), usable(found_r[i], detector, one_scale)
        good_a = planar(a, ransac) if a is not None else None
        good_b = planar(b, ransac) if b is not None else None
        if a is not None:
            objects_l.append(GRID[good_a])
            images_l.append(a[good_a])
        if b is not None:
            objects_r.append(GRID[good_b])
            images_r.append(b[good_b])
        if a is not None and b is not None and np.sum(good_a & good_b) >= 12:
            objects_s.append(GRID[good_a & good_b])
            stereo_l.append(a[good_a & good_b])
            stereo_r.append(b[good_a & good_b])
    rms_l, a1, b1, _, _ = cv2.calibrateCamera(objects_l, images_l, IMAGE_SIZE, None, None, flags=flags)
    rms_r, a2, b2, _, _ = cv2.calibrateCamera(objects_r, images_r, IMAGE_SIZE, None, None, flags=flags)
    stereo_flags = cv2.CALIB_USE_INTRINSIC_GUESS | flags if joint else cv2.CALIB_FIX_INTRINSIC
    rms_s, a1, b1, a2, b2, rot, shift, _, _ = cv2.stereoCalibrate(objects_s, stereo_l, stereo_r, a1, b1, a2, b2, IMAGE_SIZE, flags=stereo_flags)
    return (a1, b1, a2, b2, rot, shift * SQUARE_SIZE), (len(stereo_l), rms_l, rms_r, rms_s)


def rectify(cal):
    a1, b1, a2, b2, rot, shift = cal
    q1, q2, p1, p2, Q, _, _ = cv2.stereoRectify(a1, b1, a2, b2, IMAGE_SIZE, rot, shift, alpha=0.15)
    maps = (cv2.initUndistortRectifyMap(a1, b1, q1, p1, IMAGE_SIZE, cv2.CV_32FC1),
            cv2.initUndistortRectifyMap(a2, b2, q2, p2, IMAGE_SIZE, cv2.CV_32FC1))
    return q1, q2, p1, p2, Q, maps


def scene_rows(cal):
    m1, m2 = rectify(cal)[5]
    rows, disparities = [], []
    for sequence, frame in (('1_no_obstacle', 10), ('3_curved_bend', 200), ('4_circle', 400)):
        pa, pb = scene_matches(rectified(image_files(sequence, 'left')[frame], 'left', m1), rectified(image_files(sequence, 'right')[frame], 'right', m2))
        rows.append(np.abs(pa[:, 1] - pb[:, 1]))
        disparities.append(pa[:, 0] - pb[:, 0])
    rows, disparities = np.concatenate(rows), np.concatenate(disparities)
    return np.median(rows), np.percentile(rows, 90), np.mean(disparities < 0)


VARIANTS = {'classic, one scale, two-step': dict(detector='classic', one_scale=True, ransac=False, joint=False),
            'classic, two-step': dict(detector='classic', ransac=False, joint=False),
            'classic + cornerSubPix, two-step': dict(detector='subpix', ransac=False, joint=False),
            'classic, RANSAC, two-step': dict(detector='classic', joint=False),
            'classic + cornerSubPix, RANSAC, two-step': dict(detector='subpix', joint=False),
            'SB, RANSAC, two-step': dict(detector='sb', joint=False),
            'classic, RANSAC, joint': dict(detector='classic'),
            'classic + cornerSubPix, RANSAC, joint': dict(detector='subpix'),
            'SB, RANSAC, joint (used)': dict(detector='sb'),
            '  used, k3 fixed at 0': dict(detector='sb', flags=cv2.CALIB_FIX_K3),
            '  used, k3 and tangential fixed at 0': dict(detector='sb', flags=cv2.CALIB_FIX_K3 | cv2.CALIB_ZERO_TANGENT_DIST),
            '  used, only k1': dict(detector='sb', flags=cv2.CALIB_FIX_K2 | cv2.CALIB_FIX_K3 | cv2.CALIB_ZERO_TANGENT_DIST)}

start = time.time()
calibrations = {}
for name, options in VARIANTS.items():
    calibrations[name], (views, rms_l, rms_r, rms_s) = calibrate(**options)
    median, p90, negative = scene_rows(calibrations[name])
    print(f"{name:42s} views {views:2d} | rms L {rms_l:.3f} R {rms_r:.3f} stereo {rms_s:.3f} | scene rows: median {median:.2f}, 90% {p90:.2f} | negative {negative:.3f} | baseline {np.linalg.norm(calibrations[name][5]) / SQUARE_SIZE:.3f} squares")
print("time:", round(time.time() - start, 1), "s")

detector | board found: left, right, pairs | distance from the board's planar shape [px]: median, worst
  classic  | 23, 21, 20 | 0.94, 3.31
  subpix   | 23, 21, 20 | 0.19, 3.26
  sb       | 26, 25, 22 | 0.15, 0.57


classic, one scale, two-step               views 14 | rms L 1.693 R 1.230 stereo 2.380 | scene rows: median 4.63, 90% 13.36 | negative 0.424 | baseline 2.111 squares


classic, two-step                          views 20 | rms L 1.416 R 1.106 stereo 2.316 | scene rows: median 3.59, 90% 19.36 | negative 0.417 | baseline 2.148 squares


classic + cornerSubPix, two-step           views 20 | rms L 1.273 R 1.019 stereo 2.194 | scene rows: median 2.81, 90% 18.06 | negative 0.183 | baseline 2.141 squares


classic, RANSAC, two-step                  views 19 | rms L 0.343 R 0.282 stereo 0.430 | scene rows: median 0.61, 90% 1.79 | negative 0.019 | baseline 1.530 squares


classic + cornerSubPix, RANSAC, two-step   views 20 | rms L 0.144 R 0.136 stereo 0.295 | scene rows: median 0.47, 90% 1.84 | negative 0.014 | baseline 1.546 squares


... (3 outputs left out)

SB, RANSAC, joint (used)                   views 22 | rms L 0.212 R 0.164 stereo 0.217 | scene rows: median 0.19, 90% 0.61 | negative 0.009 | baseline 1.497 squares


  used, k3 fixed at 0                      views 22 | rms L 0.229 R 0.194 stereo 0.242 | scene rows: median 0.26, 90% 1.09 | negative 0.011 | baseline 1.495 squares


  used, k3 and tangential fixed at 0       views 22 | rms L 0.233 R 0.201 stereo 0.245 | scene rows: median 0.20, 90% 1.01 | negative 0.013 | baseline 1.494 squares


  used, only k1                            views 22 | rms L 0.240 R 0.202 stereo 0.255 | scene rows: median 0.18, 90% 0.64 | negative 0.014 | baseline 1.494 squares
time: 9.1 s


**3-D tests for each calibration.** Running the whole pipeline for every calibration would take minutes, so the stored robot boxes are reused. For each complete box, the bottom centre in the left image, and the same point shifted by the disparity in the right image, are mapped back to the original images with the stored calibration. These positions don't depend on any calibration. Each variant then rectifies them again, measures the disparity, and computes the 3-D position, on which the tests of part 9 of the main notebook are repeated: the angle of the straight paths of sequences 1 and 2, the circle's depth divided by its width, the speed when driving along the depth divided by the speed when driving sideways, the robot's length, and the disparity offset of the height test. The floor is fitted with `StereoBM` and RANSAC on a median of five frames per sequence, and the wheel spread is the 10-to-90 % range of the wheels' distance from it. The boxes are used directly, without the Kalman filter, so the numbers differ slightly from the main notebook's.

In [3]:
def to_raw(u, v, K, d, rect, P):
    rays = np.stack([(u - P[0, 2]) / P[0, 0], (v - P[1, 2]) / P[1, 1], np.ones_like(u)], axis=1) @ rect
    return cv2.projectPoints(rays.reshape(-1, 1, 3), np.zeros(3), np.zeros(3), K, d)[0].reshape(-1, 2)


def to_rectified(points, K, d, rect, P):
    criteria = (cv2.TERM_CRITERIA_COUNT | cv2.TERM_CRITERIA_EPS, 100, 1e-9)
    return cv2.undistortPoints(points.reshape(-1, 1, 2).astype(np.float64), K, d, None, rect, P, criteria).reshape(-1, 2)


raw, raw_backgrounds = {}, {}
for sequence in SEQUENCES:
    rows = np.array([r for r in detections[sequence] if complete(r)])
    usual = np.array([np.median(rows[max(0, k - 10):k + 11, 3]) for k in range(len(rows))])
    rows = rows[rows[:, 3] >= 0.85 * usual]
    _, x, y, w, h, d, _ = rows.T
    left, right = (K1, d1, R1, P1), (K2, d2, R2, P2)
    raw[sequence] = dict(bottom_l=to_raw(x + w / 2, y + h, *left), bottom_r=to_raw(x + w / 2 - d, y + h, *right),
                         top_l=to_raw(x + w / 2, y, *left), side_l=to_raw(x, y + h / 2, *left), side_r=to_raw(x + w, y + h / 2, *left),
                         central=(x + w / 2 > 470) & (x + w / 2 < 740))
    picks = np.linspace(0, len(image_files(sequence, 'left')) - 1, 5).astype(int)
    raw_backgrounds[sequence] = [np.median(np.stack([cv2.imread(image_files(sequence, side)[i]) for i in picks]), axis=0).astype(np.uint8)
                                 for side in ('left', 'right')]


def fit_line(ground):
    keep = np.ones(len(ground), bool)
    for _ in range(3):
        slope, intercept = np.polyfit(ground[keep, 0], ground[keep, 1], 1)
        residual = ground[:, 1] - (slope * ground[:, 0] + intercept)
        keep = np.abs(residual) < max(4 * np.median(np.abs(residual[keep])), 0.05)
    return slope


def tests_3d(cal, floor=True):
    a1, b1, a2, b2, _, _ = cal
    q1, q2, p1, p2, Q, maps = rectify(cal)
    focal, cx, cy, baseline = p1[0, 0], p1[0, 2], p1[1, 2], -p2[0, 3] / p2[0, 0]
    out, points, heights, disparities = {}, {}, [], []
    for sequence in SEQUENCES:
        r = raw[sequence]
        bottom = to_rectified(r['bottom_l'], a1, b1, q1, p1)
        d = bottom[:, 0] - to_rectified(r['bottom_r'], a2, b2, q2, p2)[:, 0]
        Z = focal * baseline / d
        points[sequence] = np.c_[(bottom[:, 0] - cx) * Z / focal, (bottom[:, 1] - cy) * Z / focal, Z]
        heights.append(bottom[:, 1] - to_rectified(r['top_l'], a1, b1, q1, p1)[:, 1])
        disparities.append(d)
        if sequence == '1_no_obstacle':
            c = r['central']
            width = to_rectified(r['side_r'][c], a1, b1, q1, p1)[:, 0] - to_rectified(r['side_l'][c], a1, b1, q1, p1)[:, 0]
            out['length'] = np.median(width * baseline / d[c])

    heights, disparities = np.concatenate(heights), np.concatenate(disparities)
    model = linear_model.RANSACRegressor(residual_threshold=8, random_state=0).fit(disparities.reshape(-1, 1), heights)
    slope, intercept = np.polyfit(disparities[model.inlier_mask_], heights[model.inlier_mask_], 1)
    out['offset'] = -intercept / slope

    wheels = np.vstack([points['3_curved_bend'], points['4_circle']])        # the floor plane through the wheels
    for _ in range(3):
        normal = np.linalg.svd(wheels - wheels.mean(axis=0))[2][2]
        distance = np.abs((wheels - wheels.mean(axis=0)) @ normal)
        wheels = wheels[distance < max(3 * np.median(distance), 0.05)]
    normal = normal if normal[1] < 0 else -normal
    side = np.array([1.0, 0, 0]) - normal[0] * normal
    side /= np.linalg.norm(side)
    ahead = np.cross(normal, side)
    ground = {s: np.c_[points[s] @ side, points[s] @ ahead] for s in SEQUENCES}

    out['angle_1'], out['angle_2'] = [np.degrees(np.arctan(fit_line(ground[s]))) for s in SEQUENCES[:2]]
    g = ground['4_circle']
    solution = np.linalg.lstsq(np.c_[2 * g, np.ones(len(g))], np.sum(g ** 2, axis=1), rcond=None)[0]
    radius = np.sqrt(solution[2] + solution[:2] @ solution[:2])
    out['circle_residual'] = (np.linalg.norm(g - solution[:2], axis=1) - radius).std()
    span = np.percentile(g, 98, axis=0) - np.percentile(g, 2, axis=0)
    out['depth/width'] = span[1] / span[0]
    step = (g[6:] - g[:-6]) / 6
    speed = np.linalg.norm(step, axis=1)
    out['speed ratio'] = np.median(speed[np.abs(step[:, 1]) > 2 * np.abs(step[:, 0])]) / np.median(speed[np.abs(step[:, 0]) > 2 * np.abs(step[:, 1])])

    if floor:
        out['floor inliers'], out['wheel spread'] = [], []
        for sequence in SEQUENCES:
            left, right = [cv2.cvtColor(cv2.remap(image, m[0], m[1], cv2.INTER_LINEAR), cv2.COLOR_BGR2GRAY)
                           for image, m in zip(raw_backgrounds[sequence], maps)]
            disparity = block_matcher.compute(left, right).astype(np.float32) / 16
            cloud = cv2.reprojectImageTo3D(disparity, Q)[500:][disparity[500:] > 5][::5]
            cloud = cloud[np.abs(cloud).max(axis=1) < 24]
            model = linear_model.RANSACRegressor(residual_threshold=0.04, random_state=0).fit(cloud[:, [0, 2]], cloud[:, 1])
            plane = np.r_[model.estimator_.coef_[0], -1, model.estimator_.coef_[1]]
            distance = (points[sequence] @ plane + model.estimator_.intercept_) / np.linalg.norm(plane)
            out['floor inliers'].append(model.inlier_mask_.mean())
            out['wheel spread'].append(np.subtract(*np.percentile(distance, [90, 10])))
    return out


print("calibration | angle seq. 1, 2 [deg] | circle depth/width, residual [m] | speed ratio | robot length [m] | offset [px] | floor inliers | wheel spread [m]")
for name in ('classic, RANSAC, two-step', 'classic + cornerSubPix, RANSAC, two-step', 'SB, RANSAC, two-step',
             'classic, RANSAC, joint', 'classic + cornerSubPix, RANSAC, joint', 'SB, RANSAC, joint (used)'):
    t = tests_3d(calibrations[name])
    print(f"  {name:42s} | {t['angle_1']:5.1f}, {t['angle_2']:5.1f} | {t['depth/width']:.2f}, {t['circle_residual']:.2f} | {t['speed ratio']:.2f} | {t['length']:.3f} | {t['offset']:5.2f}"
          f" | {np.round(t['floor inliers'], 2)} | {np.round(t['wheel spread'], 3)}")

calibration | angle seq. 1, 2 [deg] | circle depth/width, residual [m] | speed ratio | robot length [m] | offset [px] | floor inliers | wheel spread [m]
  classic, RANSAC, two-step                  |  -6.9,  -7.1 | 1.27, 0.41 | 1.08 | 0.952 |  1.24 | [0.9  0.83 0.94 0.93] | [0.079 0.216 0.187 0.243]


  classic + cornerSubPix, RANSAC, two-step   |  -2.7,  -3.6 | 1.16, 0.30 | 1.00 | 0.917 |  1.65 | [0.83 0.53 0.83 0.8 ] | [0.042 0.053 0.063 0.065]
  SB, RANSAC, two-step                       |  -1.8,  -5.1 | 1.42, 0.98 | 1.16 | 0.996 | -1.19 | [0.92 0.89 0.93 0.92] | [0.132 0.188 0.351 0.342]


  classic, RANSAC, joint                     |  -3.8,  -5.3 | 1.50, 0.82 | 1.24 | 0.981 | -0.23 | [0.06 0.14 0.06 0.06] | [0.226 0.337 0.363 0.463]
  classic + cornerSubPix, RANSAC, joint      |  -3.6,  -5.5 | 1.43, 0.71 | 1.24 | 0.976 | -0.02 | [0.89 0.65 0.86 0.79] | [0.032 0.044 0.085 0.081]


  SB, RANSAC, joint (used)                   |  -1.7,  -3.0 | 1.40, 0.63 | 1.25 | 0.964 |  0.14 | [0.98 0.88 0.9  0.96] | [0.034 0.042 0.12  0.089]


**What the row test can't see.** Each line changes one number of the used calibration by a small amount. A change that moves points up or down shows in the row test. A change that moves them along the rows hardly shows there, but it changes the depth, and with it the 3-D tests.

In [4]:
def turned(cal, axis, degrees):
    a1, b1, a2, b2, rot, shift = cal
    vector = np.zeros(3)
    vector[axis] = np.radians(degrees)
    return a1, b1, a2, b2, cv2.Rodrigues(vector)[0] @ rot, shift


used = calibrations['SB, RANSAC, joint (used)']
a1, b1, a2, b2, rot, shift = used
wider, moved = a2.copy(), a2.copy()
wider[0, 0] *= 1.005
moved[0, 2] += 5
changes = {'none': used,
           'right focal length +0.5 %': (a1, b1, wider, b2, rot, shift),
           'right image centre cx +5 px': (a1, b1, moved, b2, rot, shift),
           'turn about the vertical axis +0.2 deg': turned(used, 1, 0.2),
           'turn about the sideways axis +0.2 deg': turned(used, 0, 0.2)}
for name, cal in changes.items():
    t = tests_3d(cal, floor=False)
    print(f"{name:40s} scene rows 90%: {scene_rows(cal)[1]:.2f} px | angle seq. 1: {t['angle_1']:5.1f} deg | circle depth/width: {t['depth/width']:.2f}"
          f" | robot length: {t['length']:.3f} m | offset: {t['offset']:5.2f} px")

none                                     scene rows 90%: 0.61 px | angle seq. 1:  -1.7 deg | circle depth/width: 1.40 | robot length: 0.964 m | offset:  0.14 px


right focal length +0.5 %                scene rows 90%: 0.64 px | angle seq. 1: -11.7 deg | circle depth/width: 1.60 | robot length: 0.995 m | offset: -0.56 px


right image centre cx +5 px              scene rows 90%: 0.56 px | angle seq. 1:  -1.4 deg | circle depth/width: 0.82 | robot length: 0.758 m | offset:  5.18 px


turn about the vertical axis +0.2 deg    scene rows 90%: 0.82 px | angle seq. 1:  -0.7 deg | circle depth/width: 1.17 | robot length: 0.863 m | offset:  2.42 px


turn about the sideways axis +0.2 deg    scene rows 90%: 2.54 px | angle seq. 1:   0.3 deg | circle depth/width: 1.43 | robot length: 0.961 m | offset:  0.23 px


**Stability.** Each calibration is repeated 50 times. Each time, 37 calibration pairs are drawn at random from the 37, and each drawn pair is put back, so some pairs appear twice and others not at all. The spread of the results shows how much a variant depends on which images happen to be used. It overstates the real uncertainty, because each draw keeps only about two thirds of the different pairs, but it compares the variants fairly.

In [5]:
start = time.time()
rng = np.random.default_rng(0)
print("5 to 95 % range over 50 draws: angle seq. 1, angle seq. 2, circle depth/width, robot length, offset")
for name in ('classic, RANSAC, two-step', 'SB, RANSAC, two-step', 'classic, RANSAC, joint', 'SB, RANSAC, joint (used)'):
    results = []
    for run in range(50):
        pick = rng.integers(0, len(found_l), len(found_l))
        t = tests_3d(calibrate(**VARIANTS[name], pick=pick)[0], floor=False)
        results.append([t['angle_1'], t['angle_2'], t['depth/width'], t['length'], t['offset']])
    low, high = np.percentile(results, [5, 95], axis=0)
    print(f"  {name:28s} | {low[0]:6.1f} to {high[0]:5.1f} deg | {low[1]:6.1f} to {high[1]:5.1f} deg | {low[2]:.2f} to {high[2]:.2f}"
          f" | {low[3]:.3f} to {high[3]:.3f} m | {low[4]:5.2f} to {high[4]:5.2f} px")
print("time:", round(time.time() - start, 1), "s")

5 to 95 % range over 50 draws: angle seq. 1, angle seq. 2, circle depth/width, robot length, offset


  classic, RANSAC, two-step    |  -33.4 to   0.4 deg |  -37.4 to   5.2 deg | 0.80 to 1.83 | 0.848 to 1.105 m | -2.71 to 20.09 px


  SB, RANSAC, two-step         |  -11.0 to   7.3 deg |  -22.1 to   8.9 deg | 0.92 to 1.81 | 0.958 to 1.080 m | -2.27 to 39.03 px


  classic, RANSAC, joint       |   -4.9 to  -2.5 deg |   -7.2 to  -4.1 deg | 1.45 to 1.59 | 0.965 to 0.992 m | -0.82 to  0.36 px


  SB, RANSAC, joint (used)     |   -3.5 to  -0.7 deg |   -5.2 to  -1.9 deg | 1.35 to 1.46 | 0.958 to 0.972 m |  0.03 to  0.27 px
time: 55.5 s


#### 2. How much of the image to keep
`alpha` in `cv2.stereoRectify` sets how much of the original image stays in the rectified one. The focal length shows how much the image is zoomed. The folded share counts output pixels where the map runs backwards: the distortion formula is no longer valid there, and a mirror image of the robot appears.

In [6]:
for alpha in (0, 0.1, 0.15, 0.2, 0.25, 0.4, 0.5, 1):
    q1, q2, p1, p2, _, _, _ = cv2.stereoRectify(K1, d1, K2, d2, IMAGE_SIZE, R, T, alpha=alpha)
    shares = []
    for K, d, q, p in ((K1, d1, q1, p1), (K2, d2, q2, p2)):
        mx, my = cv2.initUndistortRectifyMap(K, d, q, p, IMAGE_SIZE, cv2.CV_32FC1)
        inside = (mx >= 0) & (mx < IMAGE_SIZE[0]) & (my >= 0) & (my < IMAGE_SIZE[1])
        folded = ((np.diff(mx, axis=1)[:-1, :] <= 0) | (np.diff(my, axis=0)[:, :-1] <= 0)) & inside[:-1, :-1]
        shares.append((inside.mean(), folded.mean()))
    print(f"alpha {alpha:<5} focal {p1[0, 0]:6.1f} px | valid: left {shares[0][0]:.3f}, right {shares[1][0]:.3f} | folded: left {shares[0][1]:.4f}, right {shares[1][1]:.4f}")

alpha 0     focal  642.3 px | valid: left 1.000, right 1.000 | folded: left 0.0000, right 0.0000
alpha 0.1   focal  619.3 px | valid: left 1.000, right 0.980 | folded: left 0.0000, right 0.0000
alpha 0.15  focal  607.7 px | valid: left 1.000, right 0.963 | folded: left 0.0000, right 0.0000
alpha 0.2   focal  596.2 px | valid: left 0.994, right 0.945 | folded: left 0.0000, right 0.0000
alpha 0.25  focal  584.6 px | valid: left 0.975, right 0.927 | folded: left 0.0000, right 0.0000
alpha 0.4   focal  550.0 px | valid: left 0.894, right 0.864 | folded: left 0.0005, right 0.0000
alpha 0.5   focal  526.9 px | valid: left 0.829, right 0.808 | folded: left 0.0063, right 0.0007
alpha 1     focal  411.5 px | valid: left 0.612, right 0.570 | folded: left 0.1090, right 0.0753


#### 3. The foreground threshold
Too low a threshold lets noise in, and too high a threshold breaks the dark robot into pieces. For each value, the cell counts on every 5th frame how many frames have a robot-sized blob, how many blobs there are per frame, and the share of the reference box that the largest blob's box covers. When the robot is whole, that share is close to 1.

In [7]:
for threshold in (8, 12, 18, 25, 40):
    seen, counts, covered = 0, [], []
    for sequence in SEQUENCES:
        rows = detections[sequence]
        for i in range(0, len(rows), 5):
            blobs = blob_list(foreground(rectified(image_files(sequence, 'left')[i], 'left'), backgrounds[sequence], threshold))
            counts.append(len(blobs))
            if blobs and blobs[0][0] >= MIN_AREA:
                seen += 1
                if complete(rows[i]):
                    covered.append(blobs[0][3] * blobs[0][4] / (rows[i, 3] * rows[i, 4]))
    print(f"threshold {threshold:2d} | frames with a robot-sized blob {seen} of {len(counts)} | blobs per frame {np.mean(counts):.1f} | largest box / reference box: median {np.median(covered):.2f}, 10% {np.percentile(covered, 10):.2f}")

threshold  8 | frames with a robot-sized blob 273 of 321 | blobs per frame 5.2 | largest box / reference box: median 1.14, 10% 1.08


threshold 12 | frames with a robot-sized blob 267 of 321 | blobs per frame 3.4 | largest box / reference box: median 1.05, 10% 1.02


threshold 18 | frames with a robot-sized blob 260 of 321 | blobs per frame 2.1 | largest box / reference box: median 1.00, 10% 0.67


threshold 25 | frames with a robot-sized blob 250 of 321 | blobs per frame 2.0 | largest box / reference box: median 0.94, 10% 0.58


threshold 40 | frames with a robot-sized blob 162 of 321 | blobs per frame 2.9 | largest box / reference box: median 0.26, 10% 0.11


#### 4. Other detectors from the course
Three simple alternatives are compared with the background subtraction on every 5th frame:
- **Colour** (Class 1): the robot's yellow panels, thresholded in HSV with hue 15 to 35. The cell prints the share of the robot's saturated pixels in that range.
- **Frame difference:** the same operation as background subtraction, but against the previous frame.
- **Optical flow** (Class 2): `cv2.calcOpticalFlowFarneback` at half size, keeping the pixels that move more than 1 px.

A hit means that the detector's box centre lies inside the reference box, and a good box means an IoU of at least 0.5 with it. A false alarm is a detection in a frame where the reference sees no robot.

In [8]:
hues = []
for sequence in SEQUENCES:
    rows = detections[sequence]
    for i in range(0, len(rows), 20):
        if complete(rows[i]):
            _, x, y, w, h = rows[i, :5].astype(int)
            crop = cv2.cvtColor(rectified(image_files(sequence, 'left')[i], 'left')[y:y + h, x:x + w], cv2.COLOR_BGR2HSV)
            hues.append(crop[(crop[:, :, 1] > 120) & (crop[:, :, 2] > 100), 0])
hues = np.concatenate(hues)
low, high = 15, 35
print("saturated pixels on the robot:", len(hues), "| share with hue 15 to 35:", round(np.mean((hues >= low) & (hues <= high)), 3))


def largest(mask):
    blobs = blob_list(cv2.dilate(cv2.erode(mask, None, iterations=1), None, iterations=2))
    return blobs[0][1:] if blobs and blobs[0][0] >= 200 else None


def colour_mask(image, previous):
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    return ((hsv[:, :, 0] >= low) & (hsv[:, :, 0] <= high) & (hsv[:, :, 1] > 120) & (hsv[:, :, 2] > 100)).astype(np.uint8) * 255


def difference_mask(image, previous):
    return (np.abs(image.astype(int) - previous.astype(int)).max(axis=2) > 18).astype(np.uint8) * 255


def flow_mask(image, previous):
    a = cv2.resize(cv2.cvtColor(previous, cv2.COLOR_BGR2GRAY), None, fx=0.5, fy=0.5)
    b = cv2.resize(cv2.cvtColor(image, cv2.COLOR_BGR2GRAY), None, fx=0.5, fy=0.5)
    flow = cv2.calcOpticalFlowFarneback(a, b, None, 0.5, 3, 15, 3, 5, 1.5, 0)
    return cv2.resize((np.linalg.norm(flow, axis=2) > 1).astype(np.uint8) * 255, IMAGE_SIZE)


for name, make_mask in (('colour', colour_mask), ('frame difference', difference_mask), ('optical flow', flow_mask)):
    start = time.time()
    hits, good, present, alarms, absent = 0, 0, 0, 0, 0
    per_sequence = {}
    for sequence in SEQUENCES:
        rows, files = detections[sequence], image_files(sequence, 'left')
        seq_hits, seq_present = 0, 0
        for i in range(5, len(rows), 5):
            box = largest(make_mask(rectified(files[i], 'left'), rectified(files[i - 1], 'left')))
            if complete(rows[i]):
                seq_present += 1
                if box is not None:
                    cx, cy = box[0] + box[2] / 2, box[1] + box[3] / 2
                    seq_hits += rows[i, 1] <= cx <= rows[i, 1] + rows[i, 3] and rows[i, 2] <= cy <= rows[i, 2] + rows[i, 4]
                    good += iou(box, rows[i, 1:5]) >= 0.5
            elif rows[i, 0] < MIN_AREA:
                absent += 1
                alarms += box is not None
        hits, present = hits + seq_hits, present + seq_present
        per_sequence[sequence[0]] = round(float(seq_hits / max(seq_present, 1)), 2)
    print(f"{name:17s} hit {hits / present:.3f} ({per_sequence}) | good box {good / present:.3f} | false alarms {alarms} of {absent} | {time.time() - start:.0f} s")

saturated pixels on the robot: 9455 | share with hue 15 to 35: 0.401


colour            hit 0.158 ({'1': 0.22, '2': 0.36, '3': 0.09, '4': 0.16}) | good box 0.000 | false alarms 0 of 59 | 15 s


frame difference  hit 0.598 ({'1': 1.0, '2': 1.0, '3': 0.59, '4': 0.48}) | good box 0.000 | false alarms 2 of 59 | 24 s


optical flow      hit 0.551 ({'1': 1.0, '2': 1.0, '3': 0.61, '4': 0.39}) | good box 0.436 | false alarms 23 of 59 | 25 s


#### 5. A classifier instead of motion: sliding window with PCA and an SVM
Background subtraction can't see a robot that stops, and a live system wouldn't have a median of the whole sequence. The sliding window of Class 8 needs neither. Here it is trained on three sequences and tested on the fourth. The boxes of the main detector are the positive samples, and random crops of the same frames are the negatives. Each window is resized to 40 × 40 pixels, reduced by `PCA`, and scored by an RBF `svm.SVC`. Non-maximum suppression with IoU then removes duplicate boxes. The lecture pairs the SVM with HOG features; this test uses plain pixels, because no course template shows HOG.

In [9]:
PATCH = (40, 40)
SIZES = (40, 56, 80, 112, 160, 224)
rng = np.random.default_rng(0)


def patch_of(gray, box):
    x, y, w, h = [int(v) for v in box]
    return cv2.resize(gray[y:y + h, x:x + w], PATCH).ravel() / 255.0


def training_set(sequences):
    samples, targets = [], []
    for sequence in sequences:
        rows, files = detections[sequence], image_files(sequence, 'left')
        for i in range(0, len(rows), 3):
            if not complete(rows[i]) or rows[i, 2] + rows[i, 4] > IMAGE_SIZE[1]:
                continue
            gray = cv2.cvtColor(rectified(files[i], 'left'), cv2.COLOR_BGR2GRAY)
            samples.append(patch_of(gray, rows[i, 1:5]))
            targets.append(1)
            size = max(rows[i, 3], rows[i, 4])
            for _ in range(3):
                candidate = (rng.integers(0, IMAGE_SIZE[0] - size), rng.integers(300, IMAGE_SIZE[1] - size), size, size)
                if iou(candidate, rows[i, 1:5]) < 0.1:
                    samples.append(patch_of(gray, candidate))
                    targets.append(0)
    return np.array(samples), np.array(targets)


def non_maximum_suppression(boxes, scores, limit=0.5):
    kept = []
    for k in np.argsort(scores)[::-1]:
        if all(iou(boxes[k], boxes[j]) <= limit for j in kept):
            kept.append(k)
    return kept


def sliding_window(gray, pca, clf):
    boxes, patches = [], []
    for size in SIZES:
        step = size // 4
        for y in range(340, IMAGE_SIZE[1] - size, step):
            for x in range(0, IMAGE_SIZE[0] - size, step):
                boxes.append((x, y, size, size))
                patches.append(cv2.resize(gray[y:y + size, x:x + size], PATCH).ravel() / 255.0)
    scores = clf.decision_function(pca.transform(np.array(patches)))
    boxes = np.array(boxes)
    above = scores > 0
    if not above.any():
        return [], []
    boxes, scores = boxes[above], scores[above]
    kept = non_maximum_suppression(boxes, scores)
    return boxes[kept], scores[kept]


for test in ('2_obstacle', '4_circle'):
    start = time.time()
    samples, targets = training_set([s for s in SEQUENCES if s != test])
    pca = PCA(n_components=40, random_state=0)
    clf = svm.SVC(kernel='rbf', C=10, gamma='scale').fit(pca.fit_transform(samples), targets)
    rows, files = detections[test], image_files(test, 'left')
    hits, good, present, extra, alarms, absent = 0, 0, 0, [], 0, 0
    for i in range(0, len(rows), 10):
        found, scores = sliding_window(cv2.cvtColor(rectified(files[i], 'left'), cv2.COLOR_BGR2GRAY), pca, clf)
        if complete(rows[i]):
            present += 1
            extra.append(max(len(found) - 1, 0))
            if len(found):
                b = found[0]
                cx, cy = b[0] + b[2] / 2, b[1] + b[3] / 2
                hits += rows[i, 1] <= cx <= rows[i, 1] + rows[i, 3] and rows[i, 2] <= cy <= rows[i, 2] + rows[i, 4]
                good += iou(b, rows[i, 1:5]) >= 0.5
        elif rows[i, 0] < MIN_AREA:
            absent += 1
            alarms += len(found) > 0
    print(f"test on {test}: {len(samples)} training samples, {np.sum(targets)} positive, explained variance {pca.explained_variance_ratio_.sum():.3f} | hit {hits / present:.3f} | good box {good / present:.3f} | other boxes per frame {np.mean(extra):.1f} | false alarms {alarms} of {absent} | {time.time() - start:.0f} s")

test on 2_obstacle: 1423 training samples, 371 positive, explained variance 0.954 | hit 1.000 | good box 0.286 | other boxes per frame 2.9 | false alarms 0 of 18 | 14 s


test on 4_circle: 666 training samples, 178 positive, explained variance 0.943 | hit 0.547 | good box 0.328 | other boxes per frame 7.9 | false alarms 1 of 1 | 16 s


#### 6. Disparity of the robot
Four ways of measuring the robot's disparity from Classes 2 and 3 are compared on every 2nd frame:
- **StereoBM:** the median of `StereoBM` over the foreground pixels.
- **Box offset:** the difference between the box centres in the left and right image.
- **Template:** `cv2.matchTemplate` of the left box along the same rows of the right image, with a parabola through the peak for sub-pixel accuracy.
- **SIFT:** the median disparity of SIFT matches inside the box, keeping pairs on nearly the same row.

In sequences 1 and 2 the distance is nearly constant, so the frame-to-frame jitter and the spread of the depth measure the noise. Sequence 4 shows the behaviour on the small, distant robot.

In [10]:
def robot_disparities(sequence, i, row):
    left, right = rectified(image_files(sequence, 'left')[i], 'left'), rectified(image_files(sequence, 'right')[i], 'right')
    gray_l, gray_r = cv2.cvtColor(left, cv2.COLOR_BGR2GRAY), cv2.cvtColor(right, cv2.COLOR_BGR2GRAY)
    x, y, w, h = row[1:5].astype(int)
    result = [row[5], np.nan, np.nan, np.nan]
    blobs = blob_list(foreground(right, backgrounds_r[sequence]))
    if blobs:
        result[1] = (x + w / 2) - (blobs[0][1] + blobs[0][3] / 2)
    x0, x1 = max(0, x - 80), min(IMAGE_SIZE[0], x + w + 20)
    scores = cv2.matchTemplate(gray_r[y:y + h, x0:x1], gray_l[y:y + h, x:x + w], cv2.TM_CCOEFF_NORMED).ravel()
    k = int(np.argmax(scores))
    shift = 0.0
    if 0 < k < len(scores) - 1:
        bend = scores[k - 1] - 2 * scores[k] + scores[k + 1]
        shift = 0.5 * (scores[k - 1] - scores[k + 1]) / bend if bend != 0 else 0.0
    result[2] = x - (x0 + k + shift)
    sift = cv2.SIFT_create()
    kp1, des1 = sift.detectAndCompute(gray_l[y:y + h, x:x + w], None)
    kp2, des2 = sift.detectAndCompute(gray_r[y:y + h, x0:x1], None)
    if des1 is not None and des2 is not None and len(kp2) >= 2:
        found = []
        for pair in cv2.BFMatcher(cv2.NORM_L2).knnMatch(des1, des2, k=2):
            if len(pair) == 2 and pair[0].distance < 0.75 * pair[1].distance:
                p, q = kp1[pair[0].queryIdx].pt, kp2[pair[0].trainIdx].pt
                if abs(p[1] - q[1]) < 2 and -2 < (p[0] + x) - (q[0] + x0) < 80:
                    found.append((p[0] + x) - (q[0] + x0))
        if len(found) >= 3:
            result[3] = np.median(found)
    return result


for sequence in ('1_no_obstacle', '2_obstacle', '4_circle'):
    rows = detections[sequence]
    table = np.array([robot_disparities(sequence, i, rows[i]) for i in range(0, len(rows), 2) if complete(rows[i])])
    print(sequence, "frames used:", len(table))
    for k, name in enumerate(('StereoBM', 'box offset', 'template', 'SIFT')):
        d = table[:, k]
        depth = FOCAL * BASELINE / np.where(d > 1, d, np.nan)
        low_z, mid_z, high_z = np.nanpercentile(depth, [10, 50, 90])
        print(f"  {name:11s} measured {np.mean(~np.isnan(d)):.2f} | disparity {np.nanmedian(d):5.1f} px | jitter {np.nanmedian(np.abs(np.diff(d))):.2f} px | depth 10/50/90 %: {low_z:.2f}/{mid_z:.2f}/{high_z:.2f} m | spread {(high_z - low_z) / mid_z:.3f}")

1_no_obstacle frames used: 57
  StereoBM    measured 1.00 | disparity  18.2 px | jitter 0.06 px | depth 10/50/90 %: 3.97/3.99/4.14 m | spread 0.043
  box offset  measured 1.00 | disparity  17.5 px | jitter 0.50 px | depth 10/50/90 %: 3.48/4.16/4.28 m | spread 0.192
  template    measured 1.00 | disparity  16.6 px | jitter 0.24 px | depth 10/50/90 %: 4.26/4.39/16.62 m | spread 2.816
  SIFT        measured 1.00 | disparity  16.8 px | jitter 0.34 px | depth 10/50/90 %: 4.16/4.33/7.89 m | spread 0.862


2_obstacle frames used: 36
  StereoBM    measured 1.00 | disparity  14.9 px | jitter 0.06 px | depth 10/50/90 %: 4.63/4.87/5.07 m | spread 0.091
  box offset  measured 1.00 | disparity  15.5 px | jitter 0.50 px | depth 10/50/90 %: 4.16/4.69/5.20 m | spread 0.221
  template    measured 1.00 | disparity   9.4 px | jitter 0.37 px | depth 10/50/90 %: 4.74/8.38/16.46 m | spread 1.400
  SIFT        measured 1.00 | disparity  13.6 px | jitter 0.44 px | depth 10/50/90 %: 4.82/5.33/11.86 m | spread 1.321


4_circle frames used: 324
  StereoBM    measured 1.00 | disparity   6.4 px | jitter 0.06 px | depth 10/50/90 %: 5.09/11.41/15.73 m | spread 0.932
  box offset  measured 1.00 | disparity   6.0 px | jitter 0.50 px | depth 10/50/90 %: 5.20/12.13/16.17 m | spread 0.905
  template    measured 1.00 | disparity   6.0 px | jitter 0.08 px | depth 10/50/90 %: 5.40/12.06/16.09 m | spread 0.886
  SIFT        measured 1.00 | disparity   6.2 px | jitter 0.08 px | depth 10/50/90 %: 5.28/11.68/15.88 m | spread 0.907
